In [5]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("meta-llama/Llama-3.2-11B-Vision-Instruct")
model = AutoModelForImageTextToText.from_pretrained("meta-llama/Llama-3.2-11B-Vision-Instruct",
                                                device_map="auto",
                                                max_memory={
                                                    0: "13GiB",
                                                    1: "13GiB",
                                                    "cpu": "30GiB"
                                                },
                                                trust_remote_code=True,
                                            )
    

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [21]:
import asyncio, json, re, logging, time

class LLM:
    def __init__(self, model="huggingface"):
        self.model = model

    def complete(self, messages):
        input_text = processor.apply_chat_template(
            messages,
            add_generation_prompt=True
        )
        inputs = processor(
            text=input_text,
            return_tensors="pt"
        ).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=512)
        response = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:])
        return re.sub(r'<\|im_end\|>', '', response)

In [7]:
llm = LLM(model)

from pydantic import BaseModel
from typing import List, Optional, Literal
from datetime import datetime

class ContentItem(BaseModel):
    type: Literal['text']  # Solo acepta 'text' como tipo
    text: str

class Message(BaseModel):
    role: Literal['system', 'user', 'assistant']  # roles comunes en chats
    content: List[ContentItem]

class ChatRequest(BaseModel):
    messages: List[Message]  # Valor por defecto lista vacía

In [20]:
from fastapi import FastAPI
from pyngrok import ngrok
import nest_asyncio
import uvicorn

ngrok.set_auth_token("344HT0PzWr1pGVLwZBa7KWXfxXE_4FMsfMKfHFpG8ZAQXrpS7")


nest_asyncio.apply()  # Permite correr uvicorn en el loop de Colab

app = FastAPI()


@app.get("/")
def home():
    return {"message": "Hola desde Colab + FastAPI!"}



@app.post("/chat")
async def chat(body: ChatRequest):
    messages = [m.model_dump() for m in body.messages]
    response = llm.complete(messages)
    
    return {
        "response": response
    }

In [ ]:
# Crear túnel en el puerto 8000
public_url = ngrok.connect(8000)
print("URL pública:", public_url)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

URL pública: NgrokTunnel: "https://transmarginally-unrebuffed-else.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4dc:e88c:7426:da:56ed:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     2803:9800:9885:a4d